In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from datasets import load_dataset
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import json

max_length = 384 # The maximum length of a feature (question and context)
doc_stride = 128 # The authorized overlap between two part of the context when splitting it is needed.

# Load the CoNLL-2003 dataset
dataset = load_dataset("squad")
train_dataset = dataset["train"]

# label_list = dataset["train"].features["ner_tags"].feature.names
# print("Categories:", label_list)
# print("Number of categories:", len(label_list))

# # Load the model and tokenizer
# model_name = "dslim/bert-base-NER"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForTokenClassification.from_pretrained(model_name)

# Load the model and tokenizer
model_name = "deepset/bert-base-cased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# # Map model labels
# labels = model.config.id2label
# print("Number of categories:", len(labels))
# print("Model labels:", labels)

importance_files = []

def evaluate_on_samples(dataset_split, num_samples):
    all_predictions = []
    all_references = []

    for i in tqdm(range(num_samples)):
        examples = dataset_split[i]
        examples["question"] = [q.lstrip() for q in examples["question"]]
        tokenized_examples = tokenizer(
            examples["question"],
            examples["context"],
            truncation="only_second",
            max_length=max_length,
            stride=doc_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
        )
        start_inx = tokenized_examples['start_positions']
        end_inx = tokenized_examples['end_positions']

        

        tokens = example["tokens"]
        ground_truth = example["ner_tags"]
        references = [label_list[tag] for tag in ground_truth]

        # print("Example:", example['tokens'])
        # print('labels:', ground_truth)
        # print("References:", references)

        # Tokenize the input tokens
        tokenized_input = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, padding=True)
        word_ids = tokenized_input.word_ids()
        # print('word_ids', word_ids)
        # print('tokenized tokens', tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'][0]))
        
        tokens_importance = {}
        
        # Map tags to subwords
        weight = 1.0
        for word_id in word_ids:
            if word_id is not None:
                # if ground_truth[word_id] in (1, 2):
                if ground_truth[word_id] == 1:
                    tokens_importance[word_id] = weight
                else:
                    tokens_importance[word_id] = 0.0

        importance_files.append(tokens_importance)
    
    # Save importance_files to a JSON file
    with open("tokens_ner_train.json", "w") as json_file:
        json.dump(importance_files, json_file, indent=4)
    print("Importance files saved to tokens_ner.json")
        

evaluate_on_samples(dataset["train"], num_samples=len(dataset["train"]))
